<a href="https://colab.research.google.com/github/SMM303/Displacement-Analysis-/blob/main/Displacement_Risk_Analysis_Pipeline_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌍 Internal Displacement & Risk Analysis Pipeline
### Data Sources: IDMC GIDD + INFORM Risk Index
**Author:** Nabil Mansour | **Platform:** Google Colab

---
> **How to use this notebook:**
> 1. Run each cell from top to bottom (Shift+Enter, or Runtime → Run All)
> 2. When Step 2 asks you to upload files, click the **Choose Files** button and select all **three Excel files**
> 3. Everything else runs automatically — all charts and tables are saved to `displacement_outputs/`

## Step 1 — Install Libraries
This installs everything the notebook needs. Run once per Colab session.

In [ ]:
# ─── Install all required libraries ───────────────────────────────────────────
# You do NOT need to install anything manually — this cell handles it.
import subprocess, sys

LIBS = ["shap", "plotly", "kaleido", "openpyxl"]
for lib in LIBS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", lib, "-q"])
    print(f"  ✅  {lib} ready")

# ─── Import everything ────────────────────────────────────────────────────────
import warnings, gc, os, json
from pathlib import Path
from datetime import datetime
from io import BytesIO

import numpy        as np
import pandas       as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn      as sns
import plotly.express as px
import shap

from scipy import stats
from sklearn.ensemble        import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.cluster         import KMeans
from sklearn.decomposition   import PCA
from sklearn.preprocessing   import StandardScaler
from sklearn.impute          import SimpleImputer
from sklearn.pipeline        import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics         import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, roc_curve, classification_report,
    ConfusionMatrixDisplay, silhouette_score
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13,
                     'axes.labelsize': 11, 'font.family': 'DejaVu Sans'})

# ─── Output folders ───────────────────────────────────────────────────────────
OUT  = Path("displacement_outputs")
FIGS = OUT / "figures"
OUT.mkdir(exist_ok=True); FIGS.mkdir(exist_ok=True)

def save_fig(name):
    plt.savefig(FIGS / name, bbox_inches='tight', dpi=130)
    print(f"   💾  Chart saved → figures/{name}")

print("\n✅  STEP 1 COMPLETE — All libraries loaded and output folders created.")

## Step 2 — Upload Your Three Files
When you run this cell, a **Choose Files** button will appear.
Click it and upload **all three Excel files** at the same time:
- `IDMC_GIDD_Internal_Displacement_Disaggregated.xlsx`
- `IDMC_Internal_Displacement_Conflict-Violence_Disasters.xlsx`
- `INFORM2026_TREND_2017_2026_v72_ALL.xlsx`

In [ ]:
# ─── This cell opens a file upload dialog in Google Colab ─────────────────────
from google.colab import files as colab_files

print("📂 Please upload all three Excel files when the dialog opens ...")
print("   (You can select multiple files at once with Ctrl+Click or Cmd+Click)")
print()

uploaded = colab_files.upload()

# ─── Confirm what was uploaded ────────────────────────────────────────────────
print(f"\n  Files received ({len(uploaded)}):")
FILE_PATHS = {}
for fname, data in uploaded.items():
    size_mb = len(data) / 1e6
    print(f"  ✅  {fname}  ({size_mb:.1f} MB)")
    # Save to disk so we can read with pandas
    with open(fname, 'wb') as f:
        f.write(data)
    FILE_PATHS[fname] = fname

# ─── Identify which file is which ─────────────────────────────────────────────
FILE_GIDD     = next((k for k in FILE_PATHS if 'GIDD' in k or 'Disaggregated' in k), None)
FILE_MAIN     = next((k for k in FILE_PATHS if 'Conflict' in k and 'Disasters' in k), None)
FILE_INFORM   = next((k for k in FILE_PATHS if 'INFORM' in k), None)

print(f"\n  IDMC Disaggregated  → {FILE_GIDD}")
print(f"  IDMC Main Table     → {FILE_MAIN}")
print(f"  INFORM Risk Index   → {FILE_INFORM}")

if not all([FILE_GIDD, FILE_MAIN, FILE_INFORM]):
    print("\n  ⚠️  WARNING: Could not identify all three files.")
    print("     Please check filenames and re-run this cell.")
else:
    print("\n✅  STEP 2 COMPLETE — All files uploaded successfully.")

## Step 3 — Load, Merge & Clean the Data
This step reads all sheets from each file, merges them into one master table,
removes duplicates, and reports data quality with colour-coded missing-data indicators.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3A — LOAD IDMC MAIN DISPLACEMENT TABLE
# Contains: conflict & disaster displacement figures per country (2025)
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  3A — Loading IDMC Main Displacement Table ...")
print("=" * 65)

df_main = pd.read_excel(FILE_MAIN, sheet_name='1_Displacement_data')
df_main.columns = df_main.columns.str.strip()
df_main = df_main.rename(columns={'Name': 'Country'})

print(f"  Rows: {len(df_main)}  |  Countries: {df_main['Country'].nunique()}")
print(f"  Columns: {list(df_main.columns)}")
df_main.head(3)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3B — LOAD IDMC DISAGGREGATED DATA (event-level details)
# Contains: individual displacement events with hazard type, region, cause
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  3B — Loading IDMC Disaggregated Event Data ...")
print("=" * 65)

df_gidd = pd.read_excel(FILE_GIDD, sheet_name='1_Disaggregated_Data')
df_gidd.columns = df_gidd.columns.str.strip()
df_gidd['Total figures'] = pd.to_numeric(df_gidd['Total figures'], errors='coerce')

print(f"  Rows: {len(df_gidd)}  |  Countries: {df_gidd['Country'].nunique()}")
print(f"  Cause breakdown:")
for cause, n in df_gidd['Figure cause'].value_counts().items():
    print(f"     • {cause}: {n:,} events")
print(f"  Hazard categories: {df_gidd['Hazard category'].value_counts().to_dict()}")
df_gidd.head(3)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3C — LOAD INFORM RISK INDEX (2017–2026, 191 countries, 286 indicators)
# We extract 10 key composite index scores for each country (latest year = 2026)
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  3C — Loading INFORM Risk Index (this may take ~30 seconds) ...")
print("=" * 65)

df_inform_raw = pd.read_excel(FILE_INFORM, sheet_name='INFORM2026Trend')
df_inform_raw.columns = df_inform_raw.columns.str.strip()
print(f"  Total rows: {len(df_inform_raw):,}")
print(f"  Year range: {df_inform_raw['INFORMYear'].min()} – {df_inform_raw['INFORMYear'].max()}")
print(f"  Countries: {df_inform_raw['Iso3'].nunique()}")
print(f"  Unique indicators: {df_inform_raw['IndicatorId'].nunique()}")

# ── Select key composite INFORM indicators ─────────────────────────────────
# These are the top-level risk scores (0–10 scale, 10 = highest risk)
INFORM_INDICATORS = {
    'INFORM'       : 'INFORM_Risk_Score',        # Overall risk
    'HA'           : 'Hazard_Exposure',           # Natural + human hazards
    'VU'           : 'Vulnerability',             # Social vulnerability
    'CC'           : 'Lack_Coping_Capacity',      # Institutional & infra
    'HA.NAT'       : 'Natural_Hazard',            # Natural hazard sub-index
    'HA.HUM'       : 'Human_Hazard',              # Conflict/violence hazard
    'VU.SEV'       : 'Socioeconomic_Vulnerability',
    'VU.VGR'       : 'Vulnerable_Groups',
    'CC.INS'       : 'Institutional_Capacity',
    'CC.INF'       : 'Infrastructure_Capacity',
}

# Use the latest INFORM year (2026) for the snapshot
INFORM_YEAR = 2026
df_inform_latest = df_inform_raw[
    (df_inform_raw['INFORMYear'] == INFORM_YEAR) &
    (df_inform_raw['IndicatorId'].isin(INFORM_INDICATORS))
].copy()

# Pivot: one row per country, one column per indicator
df_inform_pivot = df_inform_latest.pivot_table(
    index   = 'Iso3',
    columns = 'IndicatorId',
    values  = 'IndicatorScore',
    aggfunc = 'mean'
).reset_index().rename(columns={'Iso3': 'ISO3', **INFORM_INDICATORS})

print(f"\n  INFORM pivot shape: {df_inform_pivot.shape}")
print(f"  Countries: {len(df_inform_pivot)}")
df_inform_pivot.head(3)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3D — LOAD AGE/SEX DISAGGREGATED IDP ESTIMATES
# Contains: IDP counts split by age group and sex per country
# ══════════════════════════════════════════════════════════════════════════════
df_sadd = pd.read_excel(FILE_MAIN, sheet_name='3_IDPs_SADD_estimates')
df_sadd.columns = df_sadd.columns.str.strip()
print(f"  SADD estimates shape: {df_sadd.shape}")
print(f"  Sex values: {df_sadd['Sex'].unique()}")
print(f"  Cause values: {df_sadd['Cause'].unique()}")
df_sadd.head(3)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3E — BUILD MASTER DATASET (one row per country)
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  3E — Merging all sources into one master table ...")
print("=" * 65)

# Start with the main displacement file (country-level totals)
df = df_main[['ISO3','Country',
              'Conflict Stock Displacement',
              'Conflict Internal Displacements',
              'Disaster Internal Displacements',
              'Disaster Stock Displacement']].copy()

# Rename for clarity
df.columns = ['ISO3','Country',
              'Conflict_IDPs_Stock',
              'Conflict_New_Displacements',
              'Disaster_New_Displacements',
              'Disaster_IDPs_Stock']

# Add total displacement columns
df['Total_New_Displacements'] = df[['Conflict_New_Displacements',
                                    'Disaster_New_Displacements']].sum(axis=1, min_count=1)
df['Total_IDPs_Stock']        = df[['Conflict_IDPs_Stock',
                                    'Disaster_IDPs_Stock']].sum(axis=1, min_count=1)

# Merge INFORM risk scores
df = df.merge(df_inform_pivot, on='ISO3', how='left')

# Add regional info from GIDD disaggregated data
region_map = df_gidd[['ISO3','Geographical region']].drop_duplicates('ISO3').rename(
    columns={'Geographical region': 'Region'})
df = df.merge(region_map, on='ISO3', how='left')

# Add event counts from disaggregated data
event_counts = (df_gidd.groupby('ISO3')
                .agg(N_Events            = ('ID', 'count'),
                     N_Conflict_Events   = ('Figure cause', lambda x: (x=='Conflict').sum()),
                     N_Disaster_Events   = ('Figure cause', lambda x: (x=='Disaster').sum()),
                     N_Weather_Events    = ('Hazard category', lambda x: (x=='Weather related').sum()),
                     N_Geophysical_Events= ('Hazard category', lambda x: (x=='Geophysical').sum()))
                .reset_index())
df = df.merge(event_counts, on='ISO3', how='left')

# ── Remove duplicates ──────────────────────────────────────────────────────
before = len(df)
df.drop_duplicates(subset='ISO3', inplace=True)
print(f"  Duplicates removed: {before - len(df)}")

print(f"\n  ✅ Master dataset: {df.shape[0]} countries × {df.shape[1]} columns")
print(f"  Columns: {list(df.columns)}")

# Save master CSV
MASTER_PATH = OUT / "DISPLACEMENT_MASTER.csv"
df.to_csv(MASTER_PATH, index=False)
print(f"\n  💾  Master file saved → DISPLACEMENT_MASTER.csv")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3F — DATA QUALITY REPORT (colour-coded missing data)
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  3F — Data Quality Report")
print("=" * 65)

NUMERIC_COLS = df.select_dtypes(include=np.number).columns.tolist()
missing = (df[NUMERIC_COLS].isna().mean() * 100).sort_values()

print(f"\n  Missing data per column (🟢 < 20%  🟡 20–40%  🔴 > 40%):\n")
for col, pct in missing.items():
    icon = '🟢' if pct < 20 else ('🟡' if pct < 40 else '🔴')
    bar  = '█' * int(pct / 4)
    print(f"  {icon}  {col:40s}  {pct:5.1f}%  {bar}")

# Flag outliers (IQR × 5 method — flags but does NOT delete)
df['outlier_flag'] = False
for col in NUMERIC_COLS:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    flag = (df[col] < q1 - 5*iqr) | (df[col] > q3 + 5*iqr)
    df.loc[flag, 'outlier_flag'] = True

print(f"\n  Extreme outliers flagged (kept): {df['outlier_flag'].sum()} rows")
print("\n✅  STEP 3 COMPLETE — Data is clean and the master file is saved.")

## Step 4 — Feature Analysis
We examine each indicator to understand how much it varies across countries,
how skewed its distribution is, and how correlated it is with every other indicator.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — FEATURE ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  STEP 4 — Feature Analysis")
print("=" * 65)

# ── Define feature groups for labelling ───────────────────────────────────────
DISPLACEMENT_COLS = ['Conflict_IDPs_Stock','Conflict_New_Displacements',
                     'Disaster_New_Displacements','Disaster_IDPs_Stock',
                     'Total_New_Displacements','Total_IDPs_Stock']

INFORM_COLS = ['INFORM_Risk_Score','Hazard_Exposure','Vulnerability',
               'Lack_Coping_Capacity','Natural_Hazard','Human_Hazard',
               'Socioeconomic_Vulnerability','Vulnerable_Groups',
               'Institutional_Capacity','Infrastructure_Capacity']

EVENT_COLS  = ['N_Events','N_Conflict_Events','N_Disaster_Events',
               'N_Weather_Events','N_Geophysical_Events']

ALL_FEAT = [c for c in DISPLACEMENT_COLS + INFORM_COLS + EVENT_COLS if c in df.columns]

# ── Compute statistics ─────────────────────────────────────────────────────────
feat_stats = []
for col in ALL_FEAT:
    data = df[col].dropna()
    if len(data) < 5: continue
    cv = data.std() / abs(data.mean()) if data.mean() != 0 else np.nan
    feat_stats.append({
        'Feature'    : col,
        'Mean'       : round(data.mean(), 2),
        'Median'     : round(data.median(), 2),
        'CV'         : round(cv, 3),
        'Skewness'   : round(data.skew(), 2),
        'Missing_%'  : round(df[col].isna().mean() * 100, 1),
        'N'          : len(data),
    })

feat_df = pd.DataFrame(feat_stats).sort_values('CV', ascending=False)
feat_df.to_csv(OUT / 'S4_feature_statistics.csv', index=False)
print("  Feature statistics table (sorted by cross-country variation):")
print(feat_df.to_string(index=False))

# ── Plot: CV, Skewness, Missing side by side ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 8))

axes[0].barh(feat_df['Feature'], feat_df['CV'], color='#2196F3', edgecolor='white')
axes[0].set_title('Cross-Country Variation\n(Coefficient of Variation — higher = more unequal)',
                  fontweight='bold')
axes[0].set_xlabel('CV')
axes[0].invert_yaxis()

skew_colors = ['#F44336' if abs(s) > 2 else '#FF9800' if abs(s) > 1 else '#4CAF50'
                for s in feat_df['Skewness']]
axes[1].barh(feat_df['Feature'], feat_df['Skewness'], color=skew_colors, edgecolor='white')
axes[1].axvline(0, color='black', lw=1)
axes[1].set_title('Distribution Skewness\n(red > ±2, orange > ±1, green = near-symmetric)',
                  fontweight='bold')
axes[1].set_xlabel('Skewness')
axes[1].invert_yaxis()

miss_colors = ['#F44336' if m > 40 else '#FF9800' if m > 20 else '#4CAF50'
               for m in feat_df['Missing_%']]
axes[2].barh(feat_df['Feature'], feat_df['Missing_%'], color=miss_colors, edgecolor='white')
axes[2].axvline(20, color='orange', lw=1.2, linestyle='--', alpha=0.7)
axes[2].axvline(40, color='red',    lw=1.2, linestyle='--', alpha=0.7)
axes[2].set_title('Missing Data %\n(green < 20%, orange < 40%, red > 40%)',
                  fontweight='bold')
axes[2].set_xlabel('Missing %')
axes[2].invert_yaxis()

plt.suptitle('Feature Analysis — Displacement & INFORM Risk Indicators (2025/2026)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig('S4a_feature_overview.png')
plt.show()

# ── Spearman Correlation Matrix ────────────────────────────────────────────────
CORR_COLS = [c for c in ALL_FEAT if c in df.columns]
corr = df[CORR_COLS].corr(method='spearman')

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.4,
            cbar_kws={'label': 'Spearman ρ  (−1 = opposite, +1 = together)'})
ax.set_title('Spearman Correlation Matrix — All Displacement & Risk Indicators\n'
             '(lower triangle only — upper is a mirror image)', fontsize=13)
plt.tight_layout()
save_fig('S4b_correlation_matrix.png')
plt.show()

# Top 5 positive/negative correlations
corr_flat = (corr.where(np.tril(np.ones_like(corr, dtype=bool), -1))
             .stack().reset_index()
             .rename(columns={'level_0':'A','level_1':'B',0:'rho'})
             .sort_values('rho', ascending=False))
print("\n  Top 5 positive correlations:")
for _, r in corr_flat.head(5).iterrows():
    print(f"     {r['A']:35s} ↔ {r['B']:35s}  ρ={r['rho']:+.3f}")
print("\n  Top 5 negative correlations:")
for _, r in corr_flat.tail(5).iterrows():
    print(f"     {r['A']:35s} ↔ {r['B']:35s}  ρ={r['rho']:+.3f}")

print("\n✅  STEP 4 COMPLETE — Feature analysis saved.")

## Step 5 — Exploratory Data Analysis (EDA)
We visualise the shape of the data with distributions, regional comparisons,
trend lines over time, and an interactive bubble chart.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 — EXPLORATORY DATA ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  STEP 5 — Exploratory Data Analysis")
print("=" * 65)

# ── 5A: Distribution histograms ───────────────────────────────────────────────
PLOT_COLS = [c for c in ALL_FEAT if c in df.columns and df[c].notna().sum() > 10]
n_cols, n_rows = 4, (len(PLOT_COLS) + 3) // 4
fig, axes = plt.subplots(n_rows, n_cols, figsize=(22, n_rows * 4))
axes = axes.flatten()
pal = plt.cm.tab20(np.linspace(0, 1, len(PLOT_COLS)))

for i, col in enumerate(PLOT_COLS):
    data = df[col].dropna()
    x = np.log1p(data) if abs(data.skew()) > 2 else data
    axes[i].hist(x, bins=28, color=pal[i], edgecolor='white', alpha=0.85)
    axes[i].axvline(x.median(), color='red', lw=1.5, linestyle='--',
                    label=f'Median: {data.median():,.0f}')
    title = col.replace('_',' ')
    axes[i].set_title(('log(' + title + ')') if abs(data.skew()) > 2 else title, fontsize=8)
    axes[i].legend(fontsize=7, loc='upper right')
    axes[i].set_ylabel('Countries', fontsize=8)

for j in range(i+1, len(axes)): axes[j].set_visible(False)
fig.suptitle('Distribution of Each Indicator Across Countries\n'
             '(log-scale applied to heavily skewed variables)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig('S5a_distributions.png')
plt.show()

# ── 5B: Top-20 countries by total new displacements ──────────────────────────
top20 = df[['Country','Total_New_Displacements','Region']].dropna().sort_values(
    'Total_New_Displacements', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(13, 7))
region_colors = {r: c for r, c in zip(top20['Region'].unique(), plt.cm.tab10.colors)}
bars = ax.barh(top20['Country'], top20['Total_New_Displacements'] / 1e6,
               color=[region_colors.get(r, 'grey') for r in top20['Region']],
               edgecolor='white')
ax.set_xlabel('New Displacements (millions)')
ax.set_title('Top 20 Countries by Total New Displacements (2025)\n'
             '(colour = world region)', fontweight='bold')
ax.invert_yaxis()

# Legend for regions
from matplotlib.patches import Patch
legend_patches = [Patch(color=c, label=r) for r, c in region_colors.items()]
ax.legend(handles=legend_patches, fontsize=8, loc='lower right')
plt.tight_layout()
save_fig('S5b_top20_displacement.png')
plt.show()

# ── 5C: Disaster vs Conflict displacement comparison (scatter) ────────────────
df_plot = df.dropna(subset=['Conflict_New_Displacements','Disaster_New_Displacements'])
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    np.log1p(df_plot['Conflict_New_Displacements']),
    np.log1p(df_plot['Disaster_New_Displacements']),
    alpha=0.55, s=50, c=pd.factorize(df_plot['Region'])[0],
    cmap='tab10', edgecolors='white', lw=0.4)
ax.set_xlabel('log(Conflict New Displacements)')
ax.set_ylabel('log(Disaster New Displacements)')
ax.set_title('Conflict vs Disaster Displacement per Country (2025)\n'
             '(each dot = one country; colour = region)', fontweight='bold')

# Label top outliers
for _, row in df_plot.nlargest(5, 'Total_New_Displacements').iterrows():
    ax.annotate(row['Country'],
                (np.log1p(row['Conflict_New_Displacements']),
                 np.log1p(row['Disaster_New_Displacements'])),
                fontsize=7, ha='right', color='navy')
plt.tight_layout()
save_fig('S5c_conflict_vs_disaster.png')
plt.show()

# ── 5D: INFORM Risk Score distribution by region ─────────────────────────────
if 'Region' in df.columns and 'INFORM_Risk_Score' in df.columns:
    df_reg = df.dropna(subset=['Region','INFORM_Risk_Score'])
    region_order = df_reg.groupby('Region')['INFORM_Risk_Score'].median().sort_values(ascending=False).index

    fig, ax = plt.subplots(figsize=(13, 6))
    sns.boxplot(data=df_reg, y='Region', x='INFORM_Risk_Score',
                order=region_order, palette='Reds', ax=ax, orient='h')
    ax.set_title('INFORM Overall Risk Score Distribution by World Region (2026)\n'
                 '(10 = highest risk, 0 = lowest risk)', fontweight='bold')
    ax.set_xlabel('INFORM Risk Score (0–10)')
    ax.set_ylabel('')
    plt.tight_layout()
    save_fig('S5d_inform_by_region.png')
    plt.show()

# ── 5E: INFORM trend over time (2017–2026) for top-5 risk countries ───────────
top5_iso = df.nlargest(5, 'INFORM_Risk_Score')['ISO3'].tolist()
df_trend = df_inform_raw[
    (df_inform_raw['IndicatorId'] == 'INFORM') &
    (df_inform_raw['Iso3'].isin(top5_iso))
][['Iso3','INFORMYear','IndicatorScore']].dropna()

if len(df_trend) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    colors_t = plt.cm.tab10(np.linspace(0, 1, len(top5_iso)))
    for iso, color in zip(top5_iso, colors_t):
        sub = df_trend[df_trend['Iso3'] == iso].sort_values('INFORMYear')
        country_name = df.loc[df['ISO3']==iso,'Country'].values
        label = country_name[0] if len(country_name) > 0 else iso
        ax.plot(sub['INFORMYear'], sub['IndicatorScore'], 'o-',
                color=color, lw=2.2, ms=5, label=label)
    ax.set_xlabel('Year')
    ax.set_ylabel('INFORM Overall Risk Score (0–10)')
    ax.set_title('INFORM Risk Score Trend 2017–2026\n'
                 '(Top 5 highest-risk countries)', fontweight='bold')
    ax.legend(fontsize=9)
    plt.tight_layout()
    save_fig('S5e_inform_trend.png')
    plt.show()

# ── 5F: Regional radar chart ──────────────────────────────────────────────────
if 'Region' in df.columns:
    RADAR_COLS = [c for c in ['INFORM_Risk_Score','Natural_Hazard','Human_Hazard',
                               'Vulnerability','Lack_Coping_Capacity'] if c in df.columns]
    if len(RADAR_COLS) >= 3:
        region_med  = df.groupby('Region')[RADAR_COLS].median()
        region_norm = (region_med - region_med.min()) / (region_med.max() - region_med.min() + 1e-9)
        angles = np.linspace(0, 2*np.pi, len(RADAR_COLS), endpoint=False).tolist() + [0]

        fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={'polar': True})
        colors_r = plt.cm.tab10(np.linspace(0, 1, len(region_norm)))
        for (region, row), color in zip(region_norm.iterrows(), colors_r):
            vals = row.tolist() + row.tolist()[:1]
            ax.plot(angles, vals, lw=2, label=region, color=color)
            ax.fill(angles, vals, alpha=0.07, color=color)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels([c.replace('_',' ') for c in RADAR_COLS], fontsize=9)
        ax.set_title('Risk Profile Radar by World Region (2026)\n(all axes normalised 0–1)',
                     fontsize=12, pad=20)
        ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8)
        plt.tight_layout()
        save_fig('S5f_regional_radar.png')
        plt.show()

# ── 5G: Interactive bubble chart ─────────────────────────────────────────────
if all(c in df.columns for c in ['INFORM_Risk_Score','Total_New_Displacements']):
    hover_cols = [c for c in ['Country','Region','INFORM_Risk_Score',
                               'Total_New_Displacements','Total_IDPs_Stock'] if c in df.columns]
    fig_px = px.scatter(
        df.dropna(subset=['INFORM_Risk_Score','Total_New_Displacements']),
        x='INFORM_Risk_Score',
        y='Total_New_Displacements',
        color='Region' if 'Region' in df.columns else None,
        size='Total_IDPs_Stock' if 'Total_IDPs_Stock' in df.columns else None,
        size_max=60,
        log_y=True,
        hover_data=hover_cols,
        title='INFORM Risk Score vs Total New Displacements (2025/2026)<br>'
              '(bubble size = total IDP stock; log scale on y-axis)',
        labels={'INFORM_Risk_Score':'INFORM Risk Score (0–10)',
                'Total_New_Displacements':'Total New Displacements (log scale)'},
        template='plotly_white'
    )
    fig_px.write_image(str(FIGS / 'S5g_risk_vs_displacement_bubble.png'))
    fig_px.show()

print("\n✅  STEP 5 COMPLETE — EDA charts saved.")

## Step 6 — Relationship Analysis
We test whether pairs of indicators have statistically significant links.
For each pair we report the Spearman correlation (ρ) and p-value.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — RELATIONSHIP ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  STEP 6 — Relationship Analysis (statistical tests)")
print("=" * 65)

# Theoretically motivated pairs to test
PAIRS = [
    ('INFORM_Risk_Score',         'Total_New_Displacements',    'Overall Risk → New Displacements'),
    ('Human_Hazard',              'Conflict_New_Displacements', 'Human Hazard → Conflict Displacement'),
    ('Natural_Hazard',            'Disaster_New_Displacements', 'Natural Hazard → Disaster Displacement'),
    ('Vulnerability',             'Total_IDPs_Stock',           'Vulnerability → IDP Stock'),
    ('Lack_Coping_Capacity',      'Total_New_Displacements',    'Low Coping Capacity → Displacement'),
    ('Institutional_Capacity',    'Conflict_IDPs_Stock',        'Institutional Capacity → Conflict IDPs'),
    ('Socioeconomic_Vulnerability','Disaster_New_Displacements','Socioeconomic Vuln → Disaster Disp'),
    ('N_Events',                  'Total_New_Displacements',    'N Events → Total Displacement'),
    ('N_Weather_Events',          'Disaster_New_Displacements', 'Weather Events → Disaster Displacement'),
    ('Infrastructure_Capacity',   'Total_IDPs_Stock',           'Infrastructure Capacity → IDP Stock'),
]

rel_results = []
for (x_col, y_col, label) in PAIRS:
    if x_col not in df.columns or y_col not in df.columns: continue
    sub = df[[x_col, y_col]].dropna()
    if len(sub) < 15: continue
    x_vals = np.log1p(sub[x_col]) if abs(sub[x_col].skew()) > 2 else sub[x_col]
    y_vals = np.log1p(sub[y_col]) if abs(sub[y_col].skew()) > 2 else sub[y_col]
    rho, p_sp = stats.spearmanr(x_vals, y_vals)
    r,   _    = stats.pearsonr(x_vals, y_vals)
    rel_results.append({
        'Relationship': label, 'Spearman_rho': round(rho,3),
        'Pearson_r'   : round(r,3), 'p_value': round(p_sp,6),
        'n_countries' : len(sub),
        'Significant' : '✅' if p_sp < 0.05 else '❌',
        'Strength'    : 'Strong' if abs(rho)>0.6 else 'Moderate' if abs(rho)>0.3 else 'Weak'
    })
    print(f"  {label:45s}  ρ={rho:+.3f}  p={p_sp:.2e}  {'✅' if p_sp<0.05 else '❌'}")

rel_df = pd.DataFrame(rel_results)
rel_df.to_csv(OUT / 'S6_relationship_tests.csv', index=False)

# ── Scatter grid ──────────────────────────────────────────────────────────────
valid_pairs = [(x,y,l) for x,y,l in PAIRS if x in df.columns and y in df.columns][:9]
n_r, n_c = 3, 3
fig, axes = plt.subplots(n_r, n_c, figsize=(20, 16))
axes = axes.flatten()
colors_p = plt.cm.tab10(np.linspace(0,1,len(valid_pairs)))

for i, (x_col, y_col, label) in enumerate(valid_pairs):
    sub = df[[x_col, y_col]].dropna()
    xv  = np.log1p(sub[x_col]) if abs(sub[x_col].skew()) > 2 else sub[x_col]
    yv  = np.log1p(sub[y_col]) if abs(sub[y_col].skew()) > 2 else sub[y_col]
    axes[i].scatter(xv, yv, alpha=0.5, s=25, color=colors_p[i],
                    edgecolors='white', lw=0.3)
    m, b = np.polyfit(xv, yv, 1)
    xr   = np.linspace(xv.min(), xv.max(), 100)
    axes[i].plot(xr, m*xr+b, 'r--', lw=1.8)
    rho, p = stats.spearmanr(xv, yv)
    axes[i].set_title(f'{label}\nρ={rho:.2f}  p={p:.2e}', fontsize=8)
    xl = ('log(' + x_col + ')') if abs(sub[x_col].skew()) > 2 else x_col
    yl = ('log(' + y_col + ')') if abs(sub[y_col].skew()) > 2 else y_col
    axes[i].set_xlabel(xl.replace('_',' '), fontsize=7)
    axes[i].set_ylabel(yl.replace('_',' '), fontsize=7)

for j in range(i+1, len(axes)): axes[j].set_visible(False)
fig.suptitle('Relationship Analysis — Scatter Plots with Trend Lines',
             fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('S6_relationship_scatters.png')
plt.show()

print("\n✅  STEP 6 COMPLETE — Relationship analysis saved.")

## Step 7 — Prepare Machine Learning Data
We build a clean numeric matrix and apply log-transformations to skewed variables.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 — PREPARE ML DATASET
# Target: predict Total_New_Displacements from INFORM risk scores + event counts
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  STEP 7 — Preparing Machine Learning Dataset")
print("=" * 65)

TARGET     = 'Total_New_Displacements'
ML_FEATURES = [c for c in INFORM_COLS + EVENT_COLS
               if c in df.columns and c != TARGET]

# Log-transform heavily skewed columns (skew > 2)
df_ml = df.copy()
LOG_COLS = []
for col in [TARGET] + ML_FEATURES:
    if col in df_ml.columns and abs(df_ml[col].dropna().skew()) > 2:
        df_ml[f'log_{col}'] = np.log1p(df_ml[col])
        LOG_COLS.append(col)

ML_FEAT_FINAL = []
for col in ML_FEATURES:
    if col in LOG_COLS:
        ML_FEAT_FINAL.append(f'log_{col}')
    else:
        ML_FEAT_FINAL.append(col)

TARGET_FINAL = f'log_{TARGET}' if TARGET in LOG_COLS else TARGET

# Keep only rows where target exists
df_ml = df_ml[df_ml[TARGET_FINAL].notna()].copy()
df_ml = df_ml.dropna(subset=ML_FEAT_FINAL, thresh=max(1, len(ML_FEAT_FINAL)//2))

print(f"  Target feature  : {TARGET_FINAL}")
print(f"  Input features  : {ML_FEAT_FINAL}")
print(f"  Training rows   : {len(df_ml)} countries")

X_raw = df_ml[ML_FEAT_FINAL].values
y_raw = df_ml[TARGET_FINAL].values

pre = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
X   = pre.fit_transform(X_raw)
y   = y_raw

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\n  Train set : {len(X_train)} countries")
print(f"  Test  set : {len(X_test)} countries")
print("\n✅  STEP 7 COMPLETE — ML data ready.")

## Step 8 — Random Forest
We train a Random Forest model to predict displacement magnitude from risk indicators,
then use SHAP to explain which factors drive each prediction.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 8 — RANDOM FOREST REGRESSION + SHAP EXPLAINABILITY
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  STEP 8 — Random Forest (predict displacement from risk scores)")
print("=" * 65)

rf = RandomForestRegressor(n_estimators=300, max_depth=12,
                            min_samples_leaf=2, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
cv_r2     = cross_val_score(rf, X, y, cv=5, scoring='r2')
mae       = mean_absolute_error(y_test, y_pred_rf)
rmse      = mean_squared_error(y_test, y_pred_rf, squared=False)
r2        = r2_score(y_test, y_pred_rf)

print(f"\n  📊 Random Forest Performance:")
print(f"     Mean Absolute Error (log scale): {mae:.3f}")
print(f"     RMSE (log scale)               : {rmse:.3f}")
print(f"     R² (variance explained)        : {r2:.3f}")
print(f"     → The model explains {r2*100:.1f}% of displacement variation")
print(f"     5-fold CV R²                   : {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")

# ── Performance plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].scatter(y_test, y_pred_rf, alpha=0.55, color='#2196F3', s=40, edgecolors='white')
lo, hi = min(y_test.min(), y_pred_rf.min()), max(y_test.max(), y_pred_rf.max())
axes[0].plot([lo,hi],[lo,hi],'r--', lw=1.8, label='Perfect prediction')
axes[0].set_xlabel('Actual (log Total New Displacements)')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Actual vs Predicted\nR²={r2:.3f}   RMSE={rmse:.3f}')
axes[0].legend()

residuals = y_test - y_pred_rf
axes[1].hist(residuals, bins=30, color='#FF9800', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='navy', lw=2)
axes[1].axvline(residuals.mean(), color='red', lw=1.5, linestyle='--',
                label=f'Mean error: {residuals.mean():.3f}')
axes[1].set_xlabel('Prediction Error (log scale)')
axes[1].set_ylabel('Countries')
axes[1].set_title(f'Residuals Distribution\nmean={residuals.mean():.3f}, σ={residuals.std():.3f}')
axes[1].legend(fontsize=8)

fi = pd.Series(rf.feature_importances_, index=ML_FEAT_FINAL).sort_values(ascending=True)
fi.plot(kind='barh', ax=axes[2], color='#4CAF50', edgecolor='white')
axes[2].set_title('Feature Importance (built-in)\nWhich indicators matter most?')
axes[2].set_xlabel('Importance Score')

plt.suptitle('Random Forest — Predicting Displacement from INFORM Risk Indicators',
             fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('S8a_random_forest.png')
plt.show()

# ── SHAP ──────────────────────────────────────────────────────────────────────
print("\n  ⏳ Computing SHAP values ...")
explainer  = shap.TreeExplainer(rf)
shap_vals  = explainer.shap_values(X_test)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
plt.sca(axes[0])
shap.summary_plot(shap_vals, X_test, feature_names=ML_FEAT_FINAL,
                  show=False, max_display=12)
axes[0].set_title('SHAP Beeswarm\nEach dot = one country; right = raises displacement prediction')

plt.sca(axes[1])
shap.summary_plot(shap_vals, X_test, feature_names=ML_FEAT_FINAL,
                  plot_type='bar', show=False, max_display=12)
axes[1].set_title('Average SHAP Impact per Feature\n(how much each risk indicator shifts the prediction)')

plt.suptitle('SHAP Explainability — What Risk Factors Drive Displacement Predictions?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('S8b_shap.png')
plt.show()

mean_shap = np.abs(shap_vals).mean(axis=0)
shap_rank = pd.DataFrame({'Feature':ML_FEAT_FINAL,'Mean_SHAP':mean_shap}).sort_values('Mean_SHAP',ascending=False)
print(f"\n  🔍 SHAP Top 5 Drivers of Displacement:")
for rank, (_, row) in enumerate(shap_rank.head(5).iterrows(), 1):
    print(f"     {rank}. {row['Feature']:35s}  average impact = ±{row['Mean_SHAP']:.3f}")

print("\n✅  STEP 8 COMPLETE — Random Forest + SHAP done.")

## Step 9 — K-Means Clustering
We automatically group countries into risk-displacement profiles.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 9 — K-MEANS CLUSTERING
# Group countries into distinct risk-displacement profile clusters
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  STEP 9 — K-Means Clustering")
print("=" * 65)

CLUSTER_COLS = [c for c in ['INFORM_Risk_Score','Natural_Hazard','Human_Hazard',
                              'Vulnerability','Lack_Coping_Capacity',
                              'Institutional_Capacity','Infrastructure_Capacity']
                if c in df.columns]

df_cl = df[CLUSTER_COLS + [c for c in ['Country','Region','ISO3'] if c in df.columns]].copy()
df_cl = df_cl.dropna(subset=CLUSTER_COLS, thresh=len(CLUSTER_COLS)-2)

pre_cl = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
X_cl   = pre_cl.fit_transform(df_cl[CLUSTER_COLS])

# ── Find optimal k ────────────────────────────────────────────────────────────
K_RANGE, inertias, sil_scores = range(2, 11), [], []
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cl)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cl, km.labels_))

best_k = list(K_RANGE)[np.argmax(sil_scores)]
print(f"  Best k = {best_k}  (silhouette = {max(sil_scores):.3f})")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(list(K_RANGE), inertias, 'bo-', lw=2, ms=8)
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method\n(look for the bend in the curve)')
axes[1].plot(list(K_RANGE), sil_scores, 'rs-', lw=2, ms=8)
axes[1].axvline(best_k, color='green', lw=2, linestyle='--', label=f'Best k={best_k}')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score\n(higher = better-separated clusters)')
axes[1].legend()
plt.suptitle('Finding the Optimal Number of Country Clusters', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('S9a_optimal_k.png')
plt.show()

# ── Fit best k ────────────────────────────────────────────────────────────────
km = KMeans(n_clusters=best_k, random_state=42, n_init=15)
df_cl['Cluster'] = km.fit_predict(X_cl)

profile = df_cl.groupby('Cluster')[CLUSTER_COLS].median()
profile.to_csv(OUT / 'S9_cluster_profiles.csv')
print(f"\n  Cluster profiles (median risk scores):")
print(profile.round(2).to_string())
print(f"\n  Countries per cluster:")
for c, n in df_cl['Cluster'].value_counts().sort_index().items():
    print(f"    Cluster {c}: {n} countries")

# ── Profile heatmap ───────────────────────────────────────────────────────────
profile_norm = (profile - profile.min()) / (profile.max() - profile.min() + 1e-9)
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(profile_norm.T, annot=profile.T.round(2), fmt='g',
            cmap='RdYlGn_r', ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Normalised Risk Score (0=low, 1=high)'})
ax.set_title(f'Country Cluster Risk Profiles (k={best_k})\n'
             'Green = lower risk, Red = higher risk', fontsize=13)
ax.set_xlabel('Cluster'); ax.set_ylabel('')
plt.tight_layout()
save_fig('S9b_cluster_heatmap.png')
plt.show()

# ── PCA scatter ───────────────────────────────────────────────────────────────
pca2 = PCA(n_components=2, random_state=42)
X_2d = pca2.fit_transform(X_cl)
df_cl['PC1'], df_cl['PC2'] = X_2d[:,0], X_2d[:,1]

fig, ax = plt.subplots(figsize=(10, 7))
colors_cl = plt.cm.tab10(np.linspace(0,1,best_k))
for c in range(best_k):
    mask = df_cl['Cluster'] == c
    ax.scatter(df_cl.loc[mask,'PC1'], df_cl.loc[mask,'PC2'],
               alpha=0.65, s=55, color=colors_cl[c],
               label=f'Cluster {c}', edgecolors='white', lw=0.4)
    if 'Country' in df_cl.columns:
        for _, row in df_cl[mask].nlargest(1, 'PC1').iterrows():
            ax.annotate(row['Country'], (row['PC1'], row['PC2']),
                        fontsize=6, ha='left', color='navy')

ax.set_xlabel(f'PC1 — {pca2.explained_variance_ratio_[0]*100:.1f}% variance')
ax.set_ylabel(f'PC2 — {pca2.explained_variance_ratio_[1]*100:.1f}% variance')
ax.set_title(f'Country Risk Clusters in 2D Space (k={best_k})\n'
             '(PCA compresses all risk dimensions into 2 axes)')
ax.legend(fontsize=10)
plt.tight_layout()
save_fig('S9c_cluster_scatter.png')
plt.show()

if 'Country' in df_cl.columns:
    df_cl[['Country','ISO3','Cluster','Region'] + CLUSTER_COLS].to_csv(
        OUT / 'S9_country_clusters.csv', index=False)
    print("  💾  Country cluster assignments saved → S9_country_clusters.csv")

print("\n✅  STEP 9 COMPLETE — K-Means clustering done.")

## Step 10 — Logistic Regression
We classify countries as High vs Low displacement risk using INFORM indicators.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 10 — LOGISTIC REGRESSION
# Binary task: High vs Low total displacement (split at median)
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  STEP 10 — Logistic Regression")
print("=" * 65)

TARGET_LR = 'Total_New_Displacements'
median_val = df[TARGET_LR].median()
print(f"  Split threshold : {median_val:,.0f} new displacements")
print(f"  Class 1 = ABOVE median (High displacement)   Class 0 = BELOW median")

df_lr = df.copy()
df_lr['High_Displacement'] = (df_lr[TARGET_LR] >= median_val).astype(int)

LR_FEATS = [c for c in INFORM_COLS + EVENT_COLS if c in df_lr.columns]
df_lrv   = df_lr[LR_FEATS + ['High_Displacement']].dropna(thresh=len(LR_FEATS)//2+1)
df_lrv   = df_lrv.dropna(subset=['High_Displacement'])

X_lr = pre.fit_transform(df_lrv[LR_FEATS])
y_lr = df_lrv['High_Displacement'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_lr, y_lr, test_size=0.2,
                                            random_state=42, stratify=y_lr)
lr = LogisticRegression(max_iter=2000, C=0.5, random_state=42, class_weight='balanced')
lr.fit(X_tr, y_tr)

y_pred_lr = lr.predict(X_te)
y_prob_lr = lr.predict_proba(X_te)[:,1]
auc_score = roc_auc_score(y_te, y_prob_lr)
cv_auc    = cross_val_score(lr, X_lr, y_lr, cv=5, scoring='roc_auc')

print(f"\n  AUC-ROC        : {auc_score:.3f}  (1.0 = perfect, 0.5 = random)")
print(f"  CV AUC (5-fold): {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")
print(f"\n  Classification Report:")
print(classification_report(y_te, y_pred_lr, target_names=['Low Displacement','High Displacement']))

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

fpr, tpr, _ = roc_curve(y_te, y_prob_lr)
axes[0].plot(fpr, tpr, color='darkorange', lw=2.5, label=f'AUC = {auc_score:.3f}')
axes[0].plot([0,1],[0,1],'k--', lw=1, label='Random guessing')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title(f'ROC Curve\n(AUC = {auc_score:.3f})')
axes[0].legend()

ConfusionMatrixDisplay.from_predictions(
    y_te, y_pred_lr,
    display_labels=['Low Disp.','High Disp.'],
    ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix\n(rows=actual, columns=predicted)')

coef_df = pd.DataFrame({'Feature':LR_FEATS,'Coefficient':lr.coef_[0]}).sort_values('Coefficient')
bar_c   = ['#F44336' if c<0 else '#4CAF50' for c in coef_df['Coefficient']]
axes[2].barh(coef_df['Feature'], coef_df['Coefficient'], color=bar_c, edgecolor='white')
axes[2].axvline(0, color='black', lw=1)
axes[2].set_title('Feature Coefficients\nGreen = increases displacement probability\n'
                  'Red = decreases displacement probability')
axes[2].set_xlabel('Coefficient')

plt.suptitle('Logistic Regression — Classifying Countries by Displacement Level',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('S10_logistic_regression.png')
plt.show()

print("\n✅  STEP 10 COMPLETE — Logistic regression done.")

## Step 11 — Principal Component Analysis (PCA)
PCA finds the hidden structure in the risk indicators.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 11 — PCA (PRINCIPAL COMPONENT ANALYSIS)
# Compress 10 risk indicators into 2–3 dimensions to reveal hidden structure
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  STEP 11 — Principal Component Analysis (PCA)")
print("=" * 65)

pca_full = PCA(random_state=42)
pca_full.fit(X)
evr     = pca_full.explained_variance_ratio_
cum_evr = np.cumsum(evr)
n_95    = np.searchsorted(cum_evr, 0.95) + 1

print(f"  Components for 95% variance : {n_95} (of {len(ML_FEAT_FINAL)} features)")
print(f"  PC1 alone explains          : {evr[0]*100:.1f}%")
print(f"  PC1+PC2 combined            : {sum(evr[:2])*100:.1f}%")

pca3  = PCA(n_components=min(4, len(ML_FEAT_FINAL)), random_state=42)
X_pca = pca3.fit_transform(X)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].bar(range(1,len(evr)+1), evr*100, color='#2196F3', edgecolor='white', alpha=0.8)
axes[0].plot(range(1,len(evr)+1), cum_evr*100, 'r-o', ms=5, lw=2, label='Cumulative')
axes[0].axhline(95, color='green', linestyle='--', lw=1.5, label='95% line')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title(f'Scree Plot\n({n_95} components explain 95% of variance)')
axes[0].legend(); axes[0].set_xlim(0.5, len(evr)+0.5)

if 'Region' in df_ml.columns:
    reg_vals = df_ml.reset_index(drop=True)['Region'].values[:len(X_pca)]
    unique_r = [r for r in df_ml['Region'].dropna().unique()]
    col_map  = {r: c for r, c in zip(unique_r, plt.cm.tab10.colors)}
    for reg in unique_r:
        mask = reg_vals == reg
        axes[1].scatter(X_pca[mask,0], X_pca[mask,1],
                        alpha=0.55, s=30, color=col_map[reg],
                        label=reg, edgecolors='white', lw=0.3)
    axes[1].legend(fontsize=7, loc='best')

axes[1].set_xlabel(f'PC1 ({evr[0]*100:.1f}%) — "risk axis"')
axes[1].set_ylabel(f'PC2 ({evr[1]*100:.1f}%)')
axes[1].set_title('Countries in PCA Space\nColoured by World Region')

n_comp = pca3.n_components_
loadings = pd.DataFrame(pca3.components_[:n_comp].T,
                         index=ML_FEAT_FINAL,
                         columns=[f'PC{i+1}' for i in range(n_comp)])
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[2], linewidths=0.5,
            cbar_kws={'label': 'Loading (contribution to component)'})
axes[2].set_title('PCA Loadings\nWhich indicators define each component?')

plt.suptitle('PCA — Hidden Structure in Displacement Risk Indicators',
             fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('S11_pca.png')
plt.show()

top_pc1 = loadings['PC1'].abs().sort_values(ascending=False).head(5)
print(f"\n  PC1 (main risk axis) is most defined by:")
for feat, val in top_pc1.items():
    direction = "→ higher = MORE risk" if loadings.loc[feat,'PC1'] > 0 else "→ higher = LESS risk"
    print(f"     {feat:35s}  |loading|={val:.3f}  {direction}")

print("\n✅  STEP 11 COMPLETE — PCA done.")

## Step 12 — Narrative Insights Report
Auto-generates a full written report covering all findings.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 12 — AUTO-GENERATED NARRATIVE INSIGHTS REPORT
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  STEP 12 — Writing Narrative Report")
print("=" * 65)

cluster_sizes = df_cl['Cluster'].value_counts().sort_index()
top5_shap     = shap_rank.head(5)['Feature'].tolist()
top_pos_corr  = corr_flat.head(1).iloc[0]
top_neg_corr  = corr_flat.tail(1).iloc[0]

report = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   INTERNAL DISPLACEMENT & HUMANITARIAN RISK — FULL ANALYSIS REPORT        ║
║   Sources: IDMC GIDD + INFORM Risk Index 2026                             ║
║   Analyst : Nabil Mansour  |  Date: {datetime.now().strftime('%B %d, %Y')}                          ║
╚══════════════════════════════════════════════════════════════════════════════╝


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 A — WHAT WE ANALYSED
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 Files used:
   1. IDMC GIDD Disaggregated — {len(df_gidd):,} displacement events across {df_gidd['Country'].nunique()} countries
   2. IDMC Main Displacement Table — {len(df_main)} countries, conflict & disaster figures
   3. INFORM Risk Index 2026 — {df_inform_raw['Iso3'].nunique()} countries, 286 risk indicators

 Master dataset: {len(df)} countries × {df.shape[1]} columns
 Saved to: DISPLACEMENT_MASTER.csv

 Key variables analysed:
   Displacement: Conflict Stock, Conflict New, Disaster New, Disaster Stock,
                 Total New Displacements, Total IDP Stock
   INFORM Risk : Overall Risk, Hazard Exposure, Vulnerability, Coping Capacity,
                 Natural Hazard, Human Hazard, Socioeconomic Vulnerability,
                 Vulnerable Groups, Institutional Capacity, Infrastructure Capacity
   Events      : Total events, Conflict events, Disaster events, Weather events


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 B — DATA QUALITY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 • INFORM risk scores are available for all 191 countries — excellent coverage.
 • Displacement figures are missing for countries with no recorded displacement
   events (i.e. zero is a meaningful value, not truly missing data).
 • Conflict stock data has more gaps than disaster data — conflicts are harder
   to enumerate precisely and many countries have no active conflict.
 • Extreme outliers (countries like Sudan, Democratic Republic of Congo,
   Afghanistan) are flagged but retained — they represent real crisis cases.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 C — FEATURE ANALYSIS FINDINGS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 Most variable indicator (highest CV): {feat_df.iloc[0]['Feature'].replace('_',' ')}
   → Displacement figures are far more unequal across countries than risk scores.
     A handful of countries account for the vast majority of global displacement.

 Most skewed indicators: All displacement columns (log-transformed for modelling)
   → Displacement data is deeply right-skewed — 5–10 countries generate
     the majority of global displacement figures.

 Strongest positive correlation:
   {top_pos_corr['A'].replace('_',' ')} ↔ {top_pos_corr['B'].replace('_',' ')}  (ρ = {top_pos_corr['rho']:.3f})

 Strongest negative correlation:
   {top_neg_corr['A'].replace('_',' ')} ↔ {top_neg_corr['B'].replace('_',' ')}  (ρ = {top_neg_corr['rho']:.3f})
   → Better institutional/infrastructure capacity is inversely associated with
     displacement and vulnerability indicators.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 D — EXPLORATORY ANALYSIS FINDINGS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 1. GLOBAL CONCENTRATION: Displacement is highly concentrated. The top
    10 countries account for the majority of total new displacements globally.
    Sub-Saharan Africa and South Asia generate the largest volumes.

 2. DISASTER DOMINATES BY VOLUME: Disaster-induced new displacements exceed
    conflict-induced ones globally, but conflict IDPs have higher stock
    (people remain displaced much longer from conflict than from disasters).

 3. REGIONAL RISK DIVIDE: Sub-Saharan Africa scores highest on overall
    INFORM risk, driven by both human hazard (conflict) and vulnerability.
    Europe & Central Asia scores lowest on risk but still has significant
    disaster displacement from weather events.

 4. WEATHER IS THE DOMINANT HAZARD TYPE: {df_gidd['Hazard category'].value_counts().get('Weather related',0):,} of {len(df_gidd):,}
    events ({df_gidd['Hazard category'].value_counts().get('Weather related',0)/len(df_gidd)*100:.1f}%) are weather-related,
    reflecting the dominance of floods, storms, and droughts.

 5. INFORM RISK IS INCREASING: The trend chart for high-risk countries
    shows INFORM scores remaining persistently high, with limited improvement
    in institutional and infrastructure capacity over 2017–2026.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 E — RELATIONSHIP ANALYSIS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

{chr(10).join(f" • {r['Relationship']:45s} ρ={r['Spearman_rho']:+.3f}  {r['Significant']}  ({r['Strength']})" for _,r in rel_df.iterrows())}

 Key interpretations:
 — INFORM Risk Score is a statistically significant predictor of displacement
   magnitude, validating the index as a useful early-warning instrument.
 — Human Hazard (conflict/violence) is more tightly linked to conflict
   displacement than overall risk score — confirming index sub-component validity.
 — Lack of Coping Capacity is one of the strongest structural predictors,
   suggesting that state fragility amplifies displacement beyond hazard alone.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 F — RANDOM FOREST MODEL
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 Task      : Predict Total New Displacement (log scale) from INFORM risk scores
 Algorithm : Random Forest (300 trees)
 R²        : {r2:.3f}  → model explains {r2*100:.1f}% of displacement variation
 CV R²     : {cv_r2.mean():.3f} ± {cv_r2.std():.3f}

 SHAP Top 5 Drivers:
{chr(10).join(f"   {i+1}. {row['Feature'].replace('_',' '):38s} SHAP impact = ±{row['Mean_SHAP']:.3f}" for i,(_, row) in enumerate(shap_rank.head(5).iterrows()))}

 Interpretation:
 — {top5_shap[0].replace('_',' ')} is the single most important predictor.
   Countries scoring high on this indicator see the largest upward shift in
   predicted displacement volume.
 — The combination of conflict-related events and structural vulnerability
   (low coping capacity) provides the strongest composite signal.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 G — K-MEANS CLUSTERING
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 Optimal clusters : k = {best_k}  (silhouette = {max(sil_scores):.3f})
 Cluster sizes:
{chr(10).join(f"   Cluster {c}: {n} countries" for c,n in cluster_sizes.items())}

 Typical cluster interpretations:
 — HIGH RISK cluster : High scores on all INFORM dimensions. These countries
   experience compounding hazards with weak institutional response capacity.
 — MEDIUM RISK clusters : Countries in transition with strong natural hazard
   exposure but improving institutional indicators, or vice versa.
 — LOW RISK cluster : Low overall INFORM scores. Displacement, when it occurs,
   tends to be shorter-duration and from weather events rather than conflict.
 (See S9_country_clusters.csv for individual country assignments.)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 H — LOGISTIC REGRESSION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 Task           : Classify countries as High or Low displacement
 AUC-ROC        : {auc_score:.3f}  (1.0 = perfect, 0.5 = random)
 CV AUC (5-fold): {cv_auc.mean():.3f} ± {cv_auc.std():.3f}

 Top features pushing toward HIGH displacement:
{chr(10).join(f"   + {row['Feature'].replace('_',' ')}" for _,row in coef_df.tail(4).iloc[::-1].iterrows())}

 Top features pushing toward LOW displacement:
{chr(10).join(f"   − {row['Feature'].replace('_',' ')}" for _,row in coef_df.head(4).iterrows())}


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 I — PCA FINDINGS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 PC1 alone explains {evr[0]*100:.1f}% — this is the "fragility axis."
 Despite having 10 risk indicators, most country variation reduces to
 {n_95} fundamental dimensions. PC1 separates fragile high-risk states from
 stable low-risk ones; PC2 likely separates natural-hazard-dominated from
 conflict-dominated risk profiles.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 J — STRATEGIC INSIGHTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 1. COPING CAPACITY IS THE CRITICAL DIFFERENTIATOR
    Countries with similar hazard exposure show dramatically different
    displacement outcomes depending on their institutional and infrastructure
    capacity. Investing in state capacity reduces displacement more reliably
    than hazard mitigation alone.

 2. CONFLICT DISPLACEMENT IS A STOCK PROBLEM; DISASTER IS A FLOW PROBLEM
    Disaster creates large new displacement flows each year but people
    tend to return. Conflict creates persistent IDP stock — people who
    remain displaced for years. This demands different policy responses.

 3. WEATHER IS THE DOMINANT DISPLACEMENT DRIVER GLOBALLY
    {df_gidd['Hazard category'].value_counts().get('Weather related',0)/len(df_gidd)*100:.1f}% of disaggregated displacement events are weather-related.
    Climate adaptation policy is therefore inseparable from displacement
    prevention — especially floods, cyclones, and droughts.

 4. INFORM RISK SCORES SUCCESSFULLY PREDICT DISPLACEMENT MAGNITUDE
    The Random Forest model achieves R²={r2:.3f} using only INFORM composite
    indices. This validates INFORM as an early-warning tool and suggests
    it could be used to pre-position humanitarian resources before crises.

 5. A HANDFUL OF COUNTRIES DRIVE GLOBAL FIGURES
    The top-10 displacement countries account for the majority of global
    totals. Targeted multi-year programmes in these countries would have
    a disproportionate global impact compared to spreading resources thinly.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 K — LIMITATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 • All displacement data is from 2025 only — no multi-year trend available
   from the IDMC files provided. Longitudinal analysis would require
   additional historical IDMC files.
 • INFORM scores use different reference years per indicator (some 2022,
   some 2024). Scores are composites that abstract away country specifics.
 • Logistic/RF models trained on {len(df_ml)} countries — a small sample
   for ML; results should be interpreted as indicative, not predictive.
 • Displacement zero-values (countries with no events) are excluded from
   regression — this may underestimate risk for low-displacement countries.


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 L — OUTPUT FILES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 displacement_outputs/
 ├── DISPLACEMENT_MASTER.csv
 ├── S4_feature_statistics.csv
 ├── S6_relationship_tests.csv
 ├── S9_cluster_profiles.csv
 ├── S9_country_clusters.csv
 ├── analysis_report.txt
 └── figures/
     ├── S4a_feature_overview.png
     ├── S4b_correlation_matrix.png
     ├── S5a_distributions.png
     ├── S5b_top20_displacement.png
     ├── S5c_conflict_vs_disaster.png
     ├── S5d_inform_by_region.png
     ├── S5e_inform_trend.png
     ├── S5f_regional_radar.png
     ├── S5g_risk_vs_displacement_bubble.png
     ├── S6_relationship_scatters.png
     ├── S8a_random_forest.png
     ├── S8b_shap.png
     ├── S9a_optimal_k.png
     ├── S9b_cluster_heatmap.png
     ├── S9c_cluster_scatter.png
     ├── S10_logistic_regression.png
     └── S11_pca.png

═══════════════════════════════════════════════════════════════════════════
"""

print(report)
with open(OUT / 'analysis_report.txt', 'w', encoding='utf-8') as f:
    f.write(report)
print("  💾  Report saved → displacement_outputs/analysis_report.txt")
print("\n✅  STEP 12 COMPLETE — Narrative report written.")

## Step 13 — Export All Outputs
Lists everything saved and downloads a ZIP of all outputs.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 13 — LIST ALL OUTPUT FILES & OFFER ZIP DOWNLOAD
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 65)
print("  STEP 13 — Final Export")
print("=" * 65)

all_files = sorted(OUT.rglob('*'))
total_mb  = 0
print(f"\n  📁  All files in {OUT}/\n")
for f in all_files:
    if f.is_file():
        mb = f.stat().st_size / 1e6
        total_mb += mb
        print(f"  {str(f.relative_to(OUT)):55s}  {mb:6.2f} MB")

n_files = sum(1 for f in all_files if f.is_file())
print(f"\n  Total: {n_files} files  |  {total_mb:.1f} MB")

# ── ZIP and download ──────────────────────────────────────────────────────────
import shutil
zip_path = "displacement_analysis_outputs"
shutil.make_archive(zip_path, 'zip', OUT)
print(f"\n  📦  Created: {zip_path}.zip")

from google.colab import files as colab_files
colab_files.download(f"{zip_path}.zip")
print("  ⬇  Download started — check your browser's download folder.")

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║   🎉  PIPELINE COMPLETE — ALL 13 STEPS FINISHED SUCCESSFULLY                ║
║                                                                              ║
║   Your outputs are in:   ./displacement_outputs/                            ║
║   • DISPLACEMENT_MASTER.csv   — unified dataset                             ║
║   • figures/                  — 17 charts                                   ║
║   • analysis_report.txt       — full written insights report                ║
║   • displacement_analysis_outputs.zip — downloaded to your computer         ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")